In [1]:
import pandas as pd
import numpy as np
import joblib
import os

In [2]:
df = pd.read_csv('../data/Sample - Superstore.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print("Date range:", df['Order Date'].min(), "→", df['Order Date'].max())
print("Shape:", df.shape)

Date range: 2014-01-03 00:00:00 → 2017-12-30 00:00:00
Shape: (9994, 21)


In [3]:
# Group by Month — this is our forecasting target
df['Year']  = df['Order Date'].dt.year
df['Month'] = df['Order Date'].dt.month

monthly = df.groupby(['Year','Month']).agg(
    Total_Sales   = ('Sales',    'sum'),
    Total_Profit  = ('Profit',   'sum'),
    Total_Qty     = ('Quantity', 'sum'),
    Avg_Discount  = ('Discount', 'mean'),
    Order_Count   = ('Order Date', 'count')
).reset_index()

monthly = monthly.sort_values(['Year','Month']).reset_index(drop=True)
print("Monthly data shape:", monthly.shape)
monthly.head(10)

Monthly data shape: (48, 7)


,Year,Month,Total_Sales,Total_Profit,Total_Qty,Avg_Discount,Order_Count
0,2014,1,14236.8950,2450.1907,284,0.126582,79
1,2014,2,4519.8920,862.3084,159,0.176087,46
2,2014,3,55691.0090,498.7299,585,0.167516,157
3,2014,4,28295.3450,3488.8352,536,0.110000,135
4,2014,5,23648.2870,2738.7096,466,0.155328,122
5,2014,6,34595.1276,4976.5244,521,0.172000,135
6,2014,7,33946.3930,-841.4826,550,0.171678,143
7,2014,8,27909.4685,5318.1050,609,0.131046,153
8,2014,9,81777.3508,8328.0994,1000,0.159963,268
9,2014,10,31453.3930,3448.2573,573,0.160063,159


In [4]:
# Cyclical encoding of Month (captures seasonality)
monthly['Month_sin'] = np.sin(2 * np.pi * monthly['Month'] / 12)
monthly['Month_cos'] = np.cos(2 * np.pi * monthly['Month'] / 12)

# Quarter feature
monthly['Quarter'] = ((monthly['Month'] - 1) // 3) + 1

# Lag features — previous months' sales (the model learns from history)
monthly['Sales_Lag1'] = monthly['Total_Sales'].shift(1)   # 1 month ago
monthly['Sales_Lag2'] = monthly['Total_Sales'].shift(2)   # 2 months ago
monthly['Sales_Lag3'] = monthly['Total_Sales'].shift(3)   # 3 months ago

# Rolling average — smoothed trend
monthly['Sales_Roll3'] = monthly['Total_Sales'].shift(1).rolling(3).mean()

# Drop rows where lag features are NaN (first 3 rows)
monthly = monthly.dropna().reset_index(drop=True)

print("Feature-engineered shape:", monthly.shape)
monthly.head()

Feature-engineered shape: (45, 14)


,Year,Month,Total_Sales,Total_Profit,Total_Qty,Avg_Discount,Order_Count,Month_sin,Month_cos,Quarter,Sales_Lag1,Sales_Lag2,Sales_Lag3,Sales_Roll3
0,2014,4,28295.3450,3488.8352,536,0.110000,135,8.660254e-01,-0.500000,2,55691.0090,4519.8920,14236.895,24815.932000
1,2014,5,23648.2870,2738.7096,466,0.155328,122,5.000000e-01,-0.866025,2,28295.3450,55691.0090,4519.892,29502.082000
2,2014,6,34595.1276,4976.5244,521,0.172000,135,1.224647e-16,-1.000000,2,23648.2870,28295.3450,55691.009,35878.213667
3,2014,7,33946.3930,-841.4826,550,0.171678,143,-5.000000e-01,-0.866025,3,34595.1276,23648.2870,28295.345,28846.253200
4,2014,8,27909.4685,5318.1050,609,0.131046,153,-8.660254e-01,-0.500000,3,33946.3930,34595.1276,23648.287,30729.935867


In [5]:
os.makedirs('../models', exist_ok=True)
os.makedirs('../outputs', exist_ok=True)

monthly.to_csv('../data/monthly_features.csv', index=False)
print("✅ Monthly feature dataset saved!")

✅ Monthly feature dataset saved!
